In [1]:
# === RIPEMPHANTOM-T5 ===
# Comprehensive RIPEMD-160 Entropy Analysis and Non-Uniformity Research Platform
# Optimized for Google Colab

# -- Required Packages --
!pip install -q base58 pycryptodome ecdsa tqdm torch matplotlib scipy pandas seaborn statsmodels scikit-learn umap-learn

# -- Core Imports --
import os, time, json, random, hashlib, base58, math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160

# -- Statistical Analysis --
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.proportion import proportion_confint
from sklearn.cluster import DBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap

# -- Google Colab / Drive --
from google.colab import drive, files

# -- Verify scipy.stats integrity --
if not hasattr(stats, 'linregress'):
    raise ImportError("scipy.stats.linregress is not available. Check scipy installation.")

# -- Mount Drive and Prepare Workspace --
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/MyDrive/RIPEMPHANTOM-T5"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)
os.makedirs(f"{BASE_DIR}/figures", exist_ok=True)
os.makedirs(f"{BASE_DIR}/reports", exist_ok=True)
os.makedirs(f"{BASE_DIR}/datasets", exist_ok=True)

# -- Constants --
MAX_K = 2**61
ADDR_FILE = "addresses.txt"
CHECKPOINT_PATH = f"{BASE_DIR}/checkpoints"
LOG_PATH = f"{BASE_DIR}/logs/telemetry.jsonl"
REPORT_PATH = f"{BASE_DIR}/reports"
EXPECTED_PROB = 0.5  # Expected probability for uniform distribution

# -- Configuration --
RNG_SEED = int(time.time())
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# -- Save Experiment Configuration --
CONFIG = {
    "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "rng_seed": RNG_SEED,
    "max_k": MAX_K,
    "version": "RIPEMPHANTOM-T5"
}
with open(f"{BASE_DIR}/experiment_config.json", 'w') as f:
    json.dump(CONFIG, f, indent=2)

# === Block 2: Key-to-Hash160 Pipeline, Hamming Distance, Bit Flip Mapping ===

def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    """Generate RIPEMD160(SHA256(PubKey)) from a private key integer"""
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
        pubkey = prefix + vk.to_string()[:32] if compressed else b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new(); ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

def hamming_distance_and_flips(hash1: bytes, hash2: bytes) -> tuple[int, list[int]]:
    """Returns Hamming distance and list of differing bit positions"""
    distance = 0
    flips = []
    for i in range(len(hash1)):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

def key_to_vector(k: int) -> torch.Tensor:
    """Converts private key to 256-bit binary tensor"""
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32)

def hash160_to_vector(h: bytes) -> torch.Tensor:
    """Converts hash160 (20 bytes) to 160-bit binary tensor"""
    return torch.tensor([(b >> i) & 1 for b in h for i in range(7, -1, -1)], dtype=torch.float32)

def vector_to_hash160(v: torch.Tensor) -> bytes:
    """Convert a 160-bit vector back to hash160 bytes"""
    hash_bytes = bytearray(20)
    for i in range(160):
        byte_idx = i // 8
        bit_idx = 7 - (i % 8)
        if v[i] > 0.5:  # Threshold for binary decision
            hash_bytes[byte_idx] |= (1 << bit_idx)
    return bytes(hash_bytes)

def generate_batch_keys(batch_size: int) -> list[int]:
    """Generate a batch of private keys efficiently"""
    return [random.randint(1, MAX_K) for _ in range(batch_size)]

def process_key_batch(keys: list[int]) -> list[tuple[int, bytes]]:
    """Process a batch of keys to hash160, filtering invalid ones"""
    results = []
    for k in keys:
        h160 = privkey_to_hash160(k)
        if h160:
            results.append((k, h160))
    return results

# === Block 3: Neural Models for Prediction ===

class HashDistanceNet(nn.Module):
    """Predicts the Hamming distance between predicted hash160 and a target"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # Outputs predicted hamming
        )

    def forward(self, x):
        return self.model(x)

class HashPredictorNet(nn.Module):
    """Dense model that predicts hash160 bit pattern from private key"""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

class TransformerPredictor(nn.Module):
    """Transformer model to infer likely hash160 patterns from keys"""
    def __init__(self, input_size=256, output_size=160, d_model=256, nhead=8, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, 160),
            nn.Sigmoid()
        )

    def forward(self, x):
        x_proj = self.input_proj(x).unsqueeze(1)  # (B, 1, D)
        x_encoded = self.transformer(x_proj)
        out = self.output_proj(x_encoded.squeeze(1))  # (B, 160)
        return out

class EntropyAnalysisNet(nn.Module):
    """Specialized model for detecting entropy anomalies in hash160 outputs"""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(160, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 160),
            nn.Sigmoid()
        )
        self.anomaly_detector = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        anomaly_score = self.anomaly_detector(encoded)
        return decoded, anomaly_score, encoded

# === Block 4: Bit Bias Tracker, XOR Delta Memory, Mutation Log ===

bit_bias = defaultdict(float)   # Tracks effective bit flips
xor_chain = []                  # Stores successful XOR deltas
mutation_log = []               # Tracks mutation performance

def update_bit_bias(flips: list[int], reward: float = 1.0):
    """Reinforce bit positions that reduced Hamming"""
    for bit in flips:
        bit_bias[bit] += reward

def add_xor_delta(prev_k: int, new_k: int, reward: float = 1.0):
    """Register a successful XOR mutation"""
    delta = prev_k ^ new_k
    xor_chain.append((delta, reward))
    if len(xor_chain) > 100:
        xor_chain.pop(0)

def log_mutation_event(k_old, k_new, mutation_type, flips, delta_hamming):
    """Record a mutation's outcome for strategy evolution"""
    entry = {
        "time": time.time(),
        "type": mutation_type,
        "delta_hamming": delta_hamming,
        "flips": flips,
        "delta_k": k_old ^ k_new,
        "k_old": k_old,
        "k_new": k_new
    }
    mutation_log.append(entry)
    if len(mutation_log) > 5000:
        mutation_log.pop(0)

def get_top_bit_bias(n=10):
    """Return highest scoring bit positions"""
    return sorted(bit_bias.items(), key=lambda x: -x[1])[:n]

def get_top_xor_deltas(n=10):
    """Return XOR deltas with highest reward"""
    return sorted(xor_chain, key=lambda x: -x[1])[:n]

def compute_bit_bias_confidence_intervals():
    """Calculate confidence intervals for bit bias values"""
    result = {}
    total_observations = sum(bit_bias.values()) or 1  # Avoid division by zero

    for bit, value in bit_bias.items():
        proportion = value / total_observations
        lower, upper = proportion_confint(value, total_observations, alpha=0.05, method='wilson')
        result[bit] = {
            "bias": value,
            "proportion": proportion,
            "ci_lower": lower,
            "ci_upper": upper,
            "significant": lower > EXPECTED_PROB or upper < EXPECTED_PROB
        }
    return result

# === Block 5: Self-Evolving Mutation Engine ===

mutation_bank = []  # Stores (mutation_vector, success_score)

def record_mutation_result(mutation_vector: int, delta_hamming: int):
    """Track how effective a mutation was at improving entropy"""
    found = False
    for i, (vec, score) in enumerate(mutation_bank):
        if vec == mutation_vector:
            mutation_bank[i] = (vec, score + delta_hamming)
            found = True
            break
    if not found:
        mutation_bank.append((mutation_vector, delta_hamming))
    # Trim
    if len(mutation_bank) > 100:
        mutation_bank.sort(key=lambda x: -x[1])
        mutation_bank[:] = mutation_bank[:100]

def get_best_mutations(n=10):
    """Return top-performing mutation vectors"""
    return sorted(mutation_bank, key=lambda x: -x[1])[:n]

def mutate_key(k: int) -> int:
    """Create a new candidate key using all mutation intelligence"""
    new_k = k

    # Use best mutation vectors
    for vec, score in get_best_mutations(n=5):
        if random.random() < 0.3:
            new_k ^= vec

    # Flip strong bit bias positions
    for bit, score in get_top_bit_bias(n=5):
        if random.random() < 0.3:
            new_k ^= (1 << bit)

    # XOR with high-reward deltas
    for delta, reward in get_top_xor_deltas(n=5):
        if random.random() < 0.3:
            new_k ^= delta

    # Random exploratory mutation
    if random.random() < 0.2:
        new_k ^= random.getrandbits(8)

    return new_k

# === Block 6: Smart Key Generator (T5 Intelligence Blend) ===

def generate_key(seed_k: int, transformer_model=None, use_transformer=True) -> int:
    """Generates a new candidate key intelligently"""
    k = seed_k

    # Start with evolved mutations
    k = mutate_key(k)

    # Optionally apply transformer-based bit predictions
    if use_transformer and transformer_model:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(k).unsqueeze(0)
                prediction = transformer_model(k_tensor).squeeze(0)
                # Flip predicted high-value bits
                for i in range(160):
                    if prediction[i].item() > 0.95 and random.random() < 0.5:
                        k ^= (1 << (i % 160))
        except:
            pass  # fail silently if model isn't ready

    # Ensure within valid bounds
    if k < 1 or k >= MAX_K:
        k = random.randint(1, MAX_K - 1)

    return k

# === Block 7: Advanced Visualization & Statistical Analysis ===

def log_event(event_type: str, payload: dict):
    """Appends a structured JSONL event to the telemetry log"""
    entry = {
        "time": time.time(),
        "event": event_type,
        "data": payload
    }
    with open(LOG_PATH, 'a') as f:
        f.write(json.dumps(entry) + '\n')

def save_bit_bias_heatmap():
    """Visualizes which bit positions are being reinforced with confidence intervals"""
    if not bit_bias:
        return

    # Get bit bias with confidence intervals
    bias_data = compute_bit_bias_confidence_intervals()
    bits = sorted(bias_data.keys())
    values = [bias_data[bit]["bias"] for bit in bits]
    lower_ci = [bias_data[bit]["ci_lower"] for bit in bits]
    upper_ci = [bias_data[bit]["ci_upper"] for bit in bits]

    plt.figure(figsize=(16, 6))
    plt.bar(bits, values, alpha=0.7)
    plt.errorbar(bits, values, yerr=[(v-l) for v, l in zip(values, lower_ci)],
                 fmt='none', ecolor='black', capsize=3)

    # Add significance indicators
    for bit in bits:
        if bias_data[bit]["significant"]:
            plt.plot(bit, bias_data[bit]["bias"], 'r*', markersize=10)

    plt.title("Bit Bias Heatmap with 95% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Reinforcement Score")
    plt.grid(True, alpha=0.3)

    # Add reference line for expected probability
    expected_line = sum(values)/len(values) if values else 0
    plt.axhline(y=expected_line, color='r', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/figures/bit_bias_heatmap.png", dpi=300)
    plt.close()

def plot_loss_curve(losses, label: str):
    """Plots a loss curve and saves to disk"""
    if not losses:
        return
    plt.figure(figsize=(10, 6))
    plt.plot(losses)

    # Add smoothed trend line
    if len(losses) > 10:
        window_size = min(len(losses) // 10, 20)
        smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
        plt.plot(range(window_size-1, len(losses)), smoothed, 'r-', linewidth=2)

    plt.title(f"{label} Loss Curve")
    plt.xlabel("Batch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/figures/{label.lower().replace(' ', '_')}_loss.png", dpi=300)
    plt.close()

def create_bit_correlation_matrix(hash160_vectors, save_path=None):
    """Creates and visualizes correlation matrix between bit positions"""
    if not hash160_vectors:
        return None

    # Stack vectors into a matrix
    if isinstance(hash160_vectors[0], torch.Tensor):
        matrix = torch.stack(hash160_vectors).numpy()
    else:
        matrix = np.array(hash160_vectors)

    # Calculate correlation matrix
    corr_matrix = np.corrcoef(matrix.T)

    # Visualize
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.3, vmin=-0.3,
                center=0, square=True, linewidths=.5)

    plt.title('Bit Position Correlation Matrix')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

    return corr_matrix

def visualize_hash_clusters(hash_vectors, save_path=None, method='tsne'):
    """Visualize clusters in the hash space using dimensionality reduction"""
    if not hash_vectors or len(hash_vectors) < 10:
        return

    # Convert to numpy array
    if isinstance(hash_vectors[0], torch.Tensor):
        data = torch.stack(hash_vectors).numpy()
    else:
        data = np.array(hash_vectors)

    # Apply dimensionality reduction
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=RNG_SEED)
    elif method.lower() == 'umap':
        reducer = umap.UMAP(random_state=RNG_SEED)
    elif method.lower() == 'pca':
        reducer = PCA(n_components=2, random_state=RNG_SEED)
    else:
        print(f"Unknown method: {method}")
        return

    # Reduce dimensions
    reduced_data = reducer.fit_transform(data)

    # Try to find clusters
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    clusters = dbscan.fit_predict(reduced_data)

    # Plot results
    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters,
               cmap='viridis', s=50, alpha=0.7)

    # Add legend for clusters
    legend1 = plt.legend(*scatter.legend_elements(),
                        title="Clusters")
    plt.gca().add_artist(legend1)

    plt.title(f'Hash160 Clusters ({method.upper()})')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

    # Return results for further analysis
    return {
        'reduced_data': reduced_data,
        'clusters': clusters
    }

def create_3d_bit_position_map(hash160_samples, save_path=None):
    """Creates a 3D visualization of relationships between bit positions"""
    if not hash160_samples or len(hash160_samples) < 100:
        return

    # Convert to numpy arrays if needed
    if isinstance(hash160_samples[0], torch.Tensor):
        data = torch.stack(hash160_samples).numpy()
    else:
        data = np.array(hash160_samples)

    # Select 3 interesting bit positions with highest variance or bias
    bit_variance = np.var(data, axis=0)
    top_bits = np.argsort(bit_variance)[-3:]

    # Create 3D plot
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Plot points
    x = data[:, top_bits[0]]
    y = data[:, top_bits[1]]
    z = data[:, top_bits[2]]

    # Color by the sum of all bits (entropy proxy)
    colors = np.sum(data, axis=1)
    scatter = ax.scatter(x, y, z, c=colors, cmap='viridis', s=30, alpha=0.7)

    # Add colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label('Bit Sum (Entropy Proxy)')

    ax.set_xlabel(f'Bit {top_bits[0]}')
    ax.set_ylabel(f'Bit {top_bits[1]}')
    ax.set_zlabel(f'Bit {top_bits[2]}')

    plt.title('3D Relationship Between High-Variance Bit Positions')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

# === Block 8: Full Checkpointing and Auto-Resume with Enhanced State Management ===

def save_checkpoint(models: dict, step_count: int):
    """Persist models, state, bit bias, mutation memory, and progress"""
    torch.save(models['dist_net'].state_dict(), f"{CHECKPOINT_PATH}/dist_net.pt")
    torch.save(models['predictor_net'].state_dict(), f"{CHECKPOINT_PATH}/predictor_net.pt")
    torch.save(models['transformer_net'].state_dict(), f"{CHECKPOINT_PATH}/transformer_net.pt")
    torch.save(models['entropy_net'].state_dict(), f"{CHECKPOINT_PATH}/entropy_net.pt")

    # Save research state
    state = {
        "bit_bias": dict(bit_bias),
        "xor_chain": xor_chain,
        "mutation_bank": [(hex(vec), score) for vec, score in mutation_bank],
        "step": step_count,
        "timestamp": time.time(),
        "config": CONFIG
    }

    # Save compressed state file
    torch.save(state, f"{CHECKPOINT_PATH}/research_state.pt")

    # Also save individual components for backward compatibility
    with open(f"{CHECKPOINT_PATH}/bit_bias.json", 'w') as f:
        json.dump(dict(bit_bias), f)
    with open(f"{CHECKPOINT_PATH}/xor_chain.json", 'w') as f:
        json.dump(xor_chain, f)
    with open(f"{CHECKPOINT_PATH}/mutation_bank.json", 'w') as f:
        json.dump([(hex(vec), score) for vec, score in mutation_bank], f)
    with open(f"{CHECKPOINT_PATH}/step.txt", 'w') as f:
        f.write(str(step_count))

    # Log checkpoint event with metadata
    log_event("checkpoint", {
        "step": step_count,
        "timestamp": time.time(),
        "models": list(models.keys()),
        "top_bit_bias": get_top_bit_bias(5)
    })

def load_checkpoint(models: dict) -> int:
    """Load prior state if available, and return last step"""
    try:
        # Try loading consolidated state file first
        state_path = f"{CHECKPOINT_PATH}/research_state.pt"
        step = 0
        if os.path.exists(state_path):
            state = torch.load(state_path)
            bit_bias.update(state.get("bit_bias", {}))
            xor_chain.extend(state.get("xor_chain", []))
            mutation_bank.extend([(int(vec, 16), score) for vec, score in state.get("mutation_bank", [])])
            step = state.get("step", 0)

            # Update config with loaded config
            if "config" in state:
                CONFIG.update(state["config"])

            print(f"Loaded consolidated state file from step {step}")
        else:
            # Fall back to individual files
            if os.path.exists(f"{CHECKPOINT_PATH}/bit_bias.json"):
                with open(f"{CHECKPOINT_PATH}/bit_bias.json") as f:
                    bit_bias.update(json.load(f))
            if os.path.exists(f"{CHECKPOINT_PATH}/xor_chain.json"):
                with open(f"{CHECKPOINT_PATH}/xor_chain.json") as f:
                    xor_chain.extend(json.load(f))
            if os.path.exists(f"{CHECKPOINT_PATH}/mutation_bank.json"):
                with open(f"{CHECKPOINT_PATH}/mutation_bank.json") as f:
                    raw = json.load(f)
                    mutation_bank.extend([(int(vec, 16), score) for vec, score in raw])
            if os.path.exists(f"{CHECKPOINT_PATH}/step.txt"):
                with open(f"{CHECKPOINT_PATH}/step.txt") as f:
                    step = int(f.read().strip())

            print(f"Loaded available individual state files from step {step}")

        # Load model states
        if os.path.exists(f"{CHECKPOINT_PATH}/dist_net.pt"):
            models['dist_net'].load_state_dict(torch.load(f"{CHECKPOINT_PATH}/dist_net.pt"))
        if os.path.exists(f"{CHECKPOINT_PATH}/predictor_net.pt"):
            models['predictor_net'].load_state_dict(torch.load(f"{CHECKPOINT_PATH}/predictor_net.pt"))
        if os.path.exists(f"{CHECKPOINT_PATH}/transformer_net.pt"):
            models['transformer_net'].load_state_dict(torch.load(f"{CHECKPOINT_PATH}/transformer_net.pt"))

        # Try to load entropy net if it exists
        entropy_path = f"{CHECKPOINT_PATH}/entropy_net.pt"
        if os.path.exists(entropy_path):
            models['entropy_net'].load_state_dict(torch.load(entropy_path))

        log_event("checkpoint_loaded", {"step": step, "timestamp": time.time()})
        return step

    except Exception as e:
        print(f"No checkpoint loaded: {e}")
        return 0

# === Block 9: Multi-Target Support, Priority Scheduling, and Statistical Tracking ===

def load_addresses(path: str) -> list:
    """Reads address list and returns tuples (address_str, hash160_bytes)"""
    # Check for both possible file names due to Colab upload behavior
    possible_paths = [path, "addresses (1).txt", "addresses (2).txt", "addresses (3).txt",
                      "addresses (4).txt", "addresses (5).txt"]
    selected_path = None
    for p in possible_paths:
        if os.path.exists(p):
            selected_path = p
            break

    if not selected_path:
        print(f"No address file found at {path} or alternatives.")
        return []

    targets = []
    with open(selected_path, 'r') as f:
        for line in f:
            addr = line.strip()
            try:
                decoded = base58.b58decode(addr)
                h160 = decoded[1:-4]
                if len(h160) == 20:
                    targets.append((addr, h160))
            except:
                continue
    return targets

target_stats = {}  # addr_str: { best: int, best_k: int, scans: int, hamming_history: [] }

def init_target_stats(targets):
    """Initialize tracking statistics with enhanced monitoring"""
    for addr, _ in targets:
        target_stats[addr] = {
            "best": 160,
            "best_k": -1,
            "scans": 0,
            "hamming_history": [],
            "last_improvement": time.time(),
            "last_10_avg": 160
        }

def update_target_stat(addr: str, k: int, ham: int):
    """Update target statistics with enhanced metrics"""
    # Record history (limited to last 1000 points)
    if len(target_stats[addr]["hamming_history"]) >= 1000:
        target_stats[addr]["hamming_history"].pop(0)
    target_stats[addr]["hamming_history"].append(ham)

    # Update best score if improved
    if ham < target_stats[addr]["best"]:
        target_stats[addr]["best"] = ham
        target_stats[addr]["best_k"] = k
        target_stats[addr]["last_improvement"] = time.time()
        log_event("improvement", {
            "address": addr,
            "ham": ham,
            "k": k,
            "previous_best": target_stats[addr].get("best", 160)
        })

    # Update running statistics
    target_stats[addr]["scans"] += 1

    # Calculate moving average of last 10 attempts
    history = target_stats[addr]["hamming_history"]
    target_stats[addr]["last_10_avg"] = sum(history[-10:]) / min(10, len(history))

def get_priority_targets(targets, top_n=10, strategy="balanced"):
    """Return top-N most promising addresses based on advanced prioritization strategies"""
    if not target_stats:
        return random.sample(targets, min(top_n, len(targets)))

    if strategy == "balanced":
        # Balance between low Hamming and recent improvement
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"], -x[1]["last_improvement"])
        )
    elif strategy == "momentum":
        # Prioritize addresses showing recent improvement trends
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"] - x[1]["last_10_avg"], -x[1]["last_improvement"])
        )
    elif strategy == "exploration":
        # Prioritize less-scanned addresses
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["scans"], x[1]["best"])
        )
    elif strategy == "exploitation":
        # Focus purely on lowest Hamming distances
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: x[1]["best"]
        )
    else:  # Default to balanced
        sorted_targets = sorted(
            target_stats.items(),
            key=lambda x: (x[1]["best"], -x[1]["scans"])
        )

    # Get unique addresses from top targets
    top_addrs = set([addr for addr, _ in sorted_targets[:top_n]])
    return [(addr, h) for addr, h in targets if addr in top_addrs]

def analyze_target_stats():
    """Generate statistical analysis of target performance"""
    from scipy import stats  # Local import to ensure module availability

    if not target_stats:
        return {}

    analysis = {
        "global": {
            "best_hamming": min([s["best"] for s in target_stats.values()]),
            "avg_hamming": sum([s["best"] for s in target_stats.values()]) / len(target_stats),
            "total_scans": sum([s["scans"] for s in target_stats.values()]),
            "timestamp": time.time()
        },
        "targets": {}
    }

    for addr, stats in target_stats.items():
        if not stats["hamming_history"]:
            continue

        # Calculate statistical metrics
        history = stats["hamming_history"]

        analysis["targets"][addr] = {
            "best": stats["best"],
            "mean": np.mean(history),
            "median": np.median(history),
            "std_dev": np.std(history),
            "improvement_rate": (160 - stats["best"]) / max(1, stats["scans"]),
            "scans": stats["scans"]
        }

        # Calculate trend (negative means improving)
        if len(history) > 10:
            x = np.arange(len(history))
            y = np.array(history)
            try:
                print(f"Debug: Analyzing {addr}, x type: {type(x)}, y type: {type(y)}, len(x): {len(x)}, len(y): {len(y)}")
                slope, _, r_value, p_value, _ = stats.linregress(x, y)
                analysis["targets"][addr].update({
                    "trend_slope": slope,
                    "trend_r_squared": r_value**2,
                    "trend_p_value": p_value
                })
            except Exception as e:
                print(f"Error in linregress for {addr}: {e}")
                analysis["targets"][addr].update({
                    "trend_slope": 0.0,
                    "trend_r_squared": 0.0,
                    "trend_p_value": 1.0
                })

    # Save analysis
    with open(f"{BASE_DIR}/reports/target_analysis.json", 'w') as f:
        json.dump(analysis, f, indent=2)

    return analysis

# === Block 10: Main Research Loop ===

def run_phantom_t5(targets, max_steps=250_000, batch_size=64, research_mode="balanced"):
    """Main research loop for RIPEMPHANTOM-T5"""
    global bit_bias, xor_chain, mutation_bank, mutation_log

    # Initialize models
    models = {
        'dist_net': HashDistanceNet(),
        'predictor_net': HashPredictorNet(),
        'transformer_net': TransformerPredictor(),
        'entropy_net': EntropyAnalysisNet()
    }

    # Load checkpoint if available
    start_step = load_checkpoint(models)

    # Initialize optimizers
    optimizers = {
        'dist_net': torch.optim.Adam(models['dist_net'].parameters(), lr=0.001),
        'predictor_net': torch.optim.Adam(models['predictor_net'].parameters(), lr=0.001),
        'transformer_net': torch.optim.Adam(models['transformer_net'].parameters(), lr=0.001),
        'entropy_net': torch.optim.Adam(models['entropy_net'].parameters(), lr=0.001)
    }

    # Loss trackers
    losses = {
        'dist': [],
        'predictor': [],
        'transformer': [],
        'entropy': []
    }

    # Initialize target statistics
    init_target_stats(targets)

    # Tracking variables
    best_hamming = 160
    hash_samples = []
    hash_vectors = []
    anomalies_found = 0

    print(f"Starting RIPEMPHANTOM-T5 with {len(targets)} targets and {max_steps} steps...")

    # Main research loop
    try:
        for step in tqdm(range(start_step, max_steps), desc="Research Progress"):
            # Log progress every 100 steps
            if step % 100 == 0:
                print(f"Step {step}: Processed {len(hash_samples)} hash samples, Best Hamming: {best_hamming}")

            # Get priority targets based on mode
            if research_mode == "targeted":
                current_targets = get_priority_targets(targets, top_n=5, strategy="exploitation")
            elif research_mode == "entropy":
                current_targets = random.sample(targets, min(5, len(targets)))
            else:  # balanced or cluster
                current_targets = get_priority_targets(targets, top_n=10, strategy="balanced")

            # Generate batch of keys
            seed_k = random.randint(1, MAX_K)
            keys = [generate_key(seed_k, models['transformer_net'], use_transformer=(research_mode != "entropy"))
                    for _ in range(batch_size)]

            # Process keys
            batch_results = process_key_batch(keys)
            if not batch_results:
                continue

            # Convert to tensors
            key_vectors = torch.stack([key_to_vector(k) for k, _ in batch_results])
            hash_vectors_batch = [hash160_to_vector(h) for _, h in batch_results]

            # Update hash samples
            hash_samples.extend([h for _, h in batch_results])
            hash_vectors.extend(hash_vectors_batch)

            # Process each target
            for addr, target_h160 in current_targets:
                target_vector = hash160_to_vector(target_h160)

                for k, h160 in batch_results:
                    # Calculate Hamming distance
                    ham, flips = hamming_distance_and_flips(h160, target_h160)
                    update_target_stat(addr, k, ham)

                    # Update best Hamming
                    best_hamming = min(best_hamming, ham)

                    # Log improvements
                    if ham < target_stats[addr]["best"]:
                        update_bit_bias(flips, reward=1.0/ham if ham > 0 else 1.0)
                        add_xor_delta(seed_k, k, reward=1.0/ham if ham > 0 else 1.0)
                        log_mutation_event(seed_k, k, "targeted" if research_mode == "targeted" else "general",
                                         flips, ham - target_stats[addr]["best"])

            # Train models
            if research_mode != "entropy":
                # Train distance predictor
                optimizers['dist_net'].zero_grad()
                pred_hamming = models['dist_net'](key_vectors)
                true_hamming = torch.tensor([hamming_distance_and_flips(h, target_h160)[0]
                                           for _, h in batch_results], dtype=torch.float32)
                loss_dist = F.mse_loss(pred_hamming.squeeze(), true_hamming)
                loss_dist.backward()
                optimizers['dist_net'].step()
                losses['dist'].append(loss_dist.item())

                # Train hash predictor
                optimizers['predictor_net'].zero_grad()
                pred_hash = models['predictor_net'](key_vectors)
                true_hash = torch.stack(hash_vectors_batch)
                loss_pred = F.binary_cross_entropy(pred_hash, true_hash)
                loss_pred.backward()
                optimizers['predictor_net'].step()
                losses['predictor'].append(loss_pred.item())

                # Train transformer
                optimizers['transformer_net'].zero_grad()
                pred_transformer = models['transformer_net'](key_vectors)
                loss_trans = F.binary_cross_entropy(pred_transformer, true_hash)
                loss_trans.backward()
                optimizers['transformer_net'].step()
                losses['transformer'].append(loss_trans.item())

            # Train entropy analysis network
            optimizers['entropy_net'].zero_grad()
            decoded, anomaly_score, _ = models['entropy_net'](torch.stack(hash_vectors_batch))
            recon_loss = F.binary_cross_entropy(decoded, torch.stack(hash_vectors_batch))
            anomaly_loss = torch.mean(anomaly_score)  # Encourage low anomaly scores
            loss_entropy = recon_loss + 0.1 * anomaly_loss
            loss_entropy.backward()
            optimizers['entropy_net'].step()
            losses['entropy'].append(loss_entropy.item())

            # Check for anomalies
            if research_mode in ["entropy", "balanced"]:
                anomaly_scores = anomaly_score.detach().numpy()
                anomalies_found += sum(1 for score in anomaly_scores if score > 0.9)

            # Periodic visualization and analysis
            if (step + 1) % 1000 == 0:
                save_bit_bias_heatmap()
                plot_loss_curve(losses['dist'], "Distance Predictor")
                plot_loss_curve(losses['predictor'], "Hash Predictor")
                plot_loss_curve(losses['transformer'], "Transformer Predictor")
                plot_loss_curve(losses['entropy'], "Entropy Analysis")

                if research_mode in ["cluster", "balanced"]:
                    create_bit_correlation_matrix(hash_vectors[-1000:],
                                               f"{BASE_DIR}/figures/correlation_matrix_step_{step+1}.png")
                    visualize_hash_clusters(hash_vectors[-1000:],
                                         f"{BASE_DIR}/figures/hash_clusters_step_{step+1}.png",
                                         method='tsne')
                    create_3d_bit_position_map(hash_vectors[-1000:],
                                             f"{BASE_DIR}/figures/3d_bit_map_step_{step+1}.png")

                analyze_target_stats()
                save_checkpoint(models, step + 1)

    except KeyboardInterrupt:
        print(f"Interrupted at step {step}. Saving checkpoint...")
        save_checkpoint(models, step)
        print("Checkpoint saved. Exiting gracefully.")
        return {
            'best_hamming': best_hamming,
            'hash_samples': len(hash_samples),
            'anomalies_found': anomalies_found,
            'target_stats': analyze_target_stats()
        }

    # Final analysis
    save_bit_bias_heatmap()
    analyze_target_stats()
    save_checkpoint(models, max_steps)

    # Generate final visualizations
    create_bit_correlation_matrix(hash_vectors, f"{BASE_DIR}/figures/final_correlation_matrix.png")
    visualize_hash_clusters(hash_vectors, f"{BASE_DIR}/figures/final_hash_clusters.png", method='tsne')
    create_3d_bit_position_map(hash_vectors, f"{BASE_DIR}/figures/final_3d_bit_map.png")

    return {
        'best_hamming': best_hamming,
        'hash_samples': len(hash_samples),
        'anomalies_found': anomalies_found,
        'target_stats': analyze_target_stats()
    }

# === Block 12: Anomaly Detection Mode with Statistical Rigor ===

class NBISTTestSuite:
    """
    Advanced statistical test suite for cryptographic randomness assessment
    Based on NIST Statistical Test Suite framework but optimized for hash160
    """

    def __init__(self):
        self.tests = {
            "frequency": self.frequency_test,
            "frequency_block": self.block_frequency_test,
            "runs": self.runs_test,
            "longest_run": self.longest_run_test,
            "serial": self.serial_test,
            "entropy": self.approximate_entropy_test
        }

    def frequency_test(self, bits):
        """Basic frequency (monobit) test"""
        n = len(bits)
        s = sum([(2*bit-1) for bit in bits])
        s_obs = abs(s) / np.sqrt(n)
        p_value = math.erfc(s_obs / np.sqrt(2))
        return p_value

    def block_frequency_test(self, bits, block_size=10):
        """Frequency test within blocks"""
        n = len(bits)
        n_blocks = n // block_size
        if n_blocks == 0:
            return 1.0  # Not enough bits for test

        blocks = [bits[i:i+block_size] for i in range(0, n_blocks*block_size, block_size)]
        proportions = [sum(block)/block_size for block in blocks]

        chi_squared = 4.0 * block_size * sum([(prop-0.5)**2 for prop in proportions])
        p_value = stats.chi2.sf(chi_squared, n_blocks)
        return p_value

    def runs_test(self, bits):
        """Runs test (frequency of runs)"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits for meaningful test

        prop = sum(bits) / n
        tau = 2.0 / np.sqrt(n)

        if abs(prop - 0.5) >= tau:
            return 0.0  # Frequency test must be passed first

        # Count runs
        runs = 1
        for i in range(1, n):
            if bits[i] != bits[i-1]:
                runs += 1

        p_value = math.erfc(abs(runs - 2*n*prop*(1-prop)) /
                          (2*np.sqrt(2*n)*prop*(1-prop)))
        return p_value

    def longest_run_test(self, bits):
        """Test for longest run of ones"""
        n = len(bits)
        if n < 128:
            return 1.0  # Not enough bits

        # Adjust parameters based on bit length
        if n < 6272:
            k, m = 3, 8
            v = [1, 2, 3, 4]
            pi = [0.2148, 0.3672, 0.2305, 0.1875]
        else:
            k, m = 5, 128
            v = [4, 5, 6, 7, 8, 9]
            pi = [0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124]

        # Split into blocks and find longest run in each block
        num_blocks = n // m
        longest_runs = []

        for i in range(num_blocks):
            block = bits[i*m:(i+1)*m]

            # Find longest run of ones
            max_run = 0
            current_run = 0
            for bit in block:
                if bit == 1:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_run = 0

            longest_runs.append(max_run)

        # Count frequencies
        frequencies = [0] * (k+1)
        for run in longest_runs:
            if run <= v[0]:
                frequencies[0] += 1
            elif run >= v[-1]:
                frequencies[-1] += 1
            else:
                for j in range(1, len(v)):
                    if v[j-1] < run <= v[j]:
                        frequencies[j] += 1
                        break

        # Calculate chi-square
        chi_squared = sum([(frequencies[i] - num_blocks*pi[i])**2 / (num_blocks*pi[i])
                          for i in range(len(frequencies))])

        p_value = stats.chi2.sf(chi_squared, k)
        return p_value

    def serial_test(self, bits, m=3):
        """Serial test (uniformity of m-bit patterns)"""
        n = len(bits)
        if n < 2**m:
            return 1.0  # Not enough bits

        # Extend bits for wrapping
        extended_bits = bits + bits[:m-1]

        # Count m-bit patterns
        pattern_counts1 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m])
            pattern_counts1[pattern] += 1

        # Count (m-1)-bit patterns
        pattern_counts2 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-1])
            pattern_counts2[pattern] += 1

        # Count (m-2)-bit patterns
        pattern_counts3 = defaultdict(int)
        for i in range(n):
            pattern = tuple(extended_bits[i:i+m-2])
            pattern_counts3[pattern] += 1

        # Calculate test statistics
        psi_sq_m = sum([count**2 for count in pattern_counts1.values()]) * 2**m / n - n
        psi_sq_m1 = sum([count**2 for count in pattern_counts2.values()]) * 2**(m-1) / n - n
        psi_sq_m2 = sum([count**2 for count in pattern_counts3.values()]) * 2**(m-2) / n - n

        delta1 = psi_sq_m - psi_sq_m1
        delta2 = psi_sq_m - 2*psi_sq_m1 + psi_sq_m2

        p_value1 = stats.chi2.sf(delta1, 2**(m-1))
        p_value2 = stats.chi2.sf(delta2, 2**(m-2))

        return min(p_value1, p_value2)

    def approximate_entropy_test(self, bits, m=5):
        """Approximate entropy test"""
        n = len(bits)
        if n < 100:
            return 1.0  # Not enough bits

        # Function to compute frequency of overlapping blocks
        def phi(m_val):
            # Count frequencies of m-bit patterns
            counts = defaultdict(int)
            for i in range(n):
                # Get the pattern using modulo to handle wrap-around
                pattern = tuple(bits[(i+j) % n] for j in range(m_val))
                counts[pattern] += 1

            # Compute C_m^i
            c_m = [count/n for count in counts.values()]
            return sum([p * np.log(p) for p in c_m if p > 0])

        # Compute ApEn
        apen = phi(m) - phi(m+1)
        chi_squared = 2 * n * (np.log(2) - apen)

        # Compute p-value (degrees of freedom = 2^m)
        p_value = stats.chi2.sf(chi_squared, 2**m)
        return p_value

    def run_all_tests(self, bits):
        """Run all tests and return results with confidence intervals"""
        results = {}
        for test_name, test_func in self.tests.items():
            p_value = test_func(bits)
            passed = p_value >= 0.01  # Standard threshold

            # Using Wilson score interval for pass/fail confidence
            conf_level = 0.95
            z = stats.norm.ppf(1 - (1 - conf_level) / 2)

            # For pass/fail binary outcome
            if passed:
                lower_ci = (1 + 1/(4*1) - z/(2*np.sqrt(1)))
                upper_ci = 1.0
            else:
                lower_ci = 0.0
                upper_ci = z/(2*np.sqrt(1))

            results[test_name] = {
                "p_value": p_value,
                "passed": passed,
                "confidence_lower": lower_ci,
                "confidence_upper": upper_ci
            }

        return results

def run_entropy_scan(sample_size: int = 100_000, reference_hashes: list = [],
                    statistical_rigor: bool = True):
    """Scan keyspace and map entropy and/or Hamming distributions with statistical rigor"""
    global entropy_counter, entropy_total, entropy_samples

    print(f"Running anomaly detection scan with {sample_size:,} keys...")

    # Reset counters
    entropy_counter = np.zeros(160, dtype=int)
    entropy_total = 0
    entropy_samples = 0
    hamming_distribution = []

    # Collect hash samples for statistical analysis
    hash_samples = []
    bit_vectors = []

    for _ in tqdm(range(sample_size), desc="Entropy Scan"):
        k = random.randint(1, MAX_K)
        h160 = privkey_to_hash160(k)
        if not h160:
            continue

        # Record entropy
        entropy_samples += 1
        for i in range(20):
            byte = h160[i]
            for j in range(8):
                bit_index = (i * 8) + (7 - j)
                if byte & (1 << j):
                    entropy_counter[bit_index] += 1

        entropy_total += 1

        # Collect sample for further analysis
        hash_samples.append(h160)
        bit_vectors.append(hash160_to_vector(h160).numpy())

        # Track Hamming distances to reference targets
        for _, ref in reference_hashes:
            dist, _ = hamming_distance_and_flips(h160, ref)
            hamming_distribution.append(dist)

    # --- Basic Visualization ---

    # Plot entropy bias with confidence intervals
    plt.figure(figsize=(16, 6))

    # Calculate proportion and confidence intervals
    proportions = entropy_counter / max(1, entropy_samples)

    # Wilson score interval for confidence bounds
    confidence = 0.95
    z = stats.norm.ppf(1 - (1 - confidence) / 2)

    lower_ci = []
    upper_ci = []

    for i in range(160):
        p = proportions[i]
        n = entropy_samples

        # Wilson score interval
        denominator = 1 + z**2/n
        centre_adjusted_prob = (p + z**2/(2*n))/denominator
        adjusted_standard_dev = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))/denominator

        lower = max(0, centre_adjusted_prob - adjusted_standard_dev)
        upper = min(1, centre_adjusted_prob + adjusted_standard_dev)

        lower_ci.append(lower)
        upper_ci.append(upper)

    # Plot with confidence intervals
    plt.bar(range(160), proportions, alpha=0.7)
    plt.errorbar(range(160), proportions,
                yerr=[proportions-lower_ci, upper_ci-proportions],
                fmt='none', ecolor='black', capsize=2, alpha=0.3)

    # Add reference line for expected probability
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Expected (0.5)')

    # Add significance markers
    for i in range(160):
        # If confidence interval doesn't include 0.5, it's significant
        if lower_ci[i] > 0.5 or upper_ci[i] < 0.5:
            plt.plot(i, proportions[i], 'r*', markersize=8)

    plt.title(f"RIPEMD-160 Bitwise Distribution with {confidence*100:.0f}% Confidence Intervals")
    plt.xlabel("Bit Position (0–159)")
    plt.ylabel("Frequency of 1s")
    plt.ylim(0.45, 0.55)  # Zoom in around 0.5
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/figures/hash160_entropy_profile.png", dpi=300)
    plt.close()

    # Plot Hamming distribution
    if hamming_distribution:
        plt.figure(figsize=(12, 8))

        # Get data for distribution
        hist, bin_edges = np.histogram(hamming_distribution, bins=40)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Plot histogram
        plt.bar(bin_centers, hist, width=(bin_edges[1]-bin_edges[0])*0.8,
               color='steelblue', edgecolor='black', alpha=0.7)

        # Add expected distribution for comparison
        # For perfectly random bit-flips, expect binomial distribution with p=0.5
        x = np.arange(40, 120)
        n = 160  # 160 bits
        p = 0.5  # probability of each bit being different
        expected = stats.binom.pmf(x, n, p) * sum(hist) * (bin_edges[1]-bin_edges[0])
        plt.plot(x, expected, 'r-', linewidth=2,
                label='Expected (Binomial Distribution)')

        # Add mean and standard deviation
        mean_ham = np.mean(hamming_distribution)
        std_ham = np.std(hamming_distribution)
        expected_mean = n * p
        expected_std = np.sqrt(n * p * (1-p))

        plt.axvline(x=mean_ham, color='blue', linestyle='--',
                   label=f'Observed Mean: {mean_ham:.2f}')
        plt.axvline(x=expected_mean, color='red', linestyle='--',
                   label=f'Expected Mean: {expected_mean:.2f}')

        # Add KS-test for goodness of fit
        # Normalize data for KS test
        ks_stat, p_value = stats.kstest(
            hamming_distribution,
            stats.binom.cdf,
            args=(n, p)
        )

        plt.title(f'Hamming Distances to Reference Hash160s\nKS-test: D={ks_stat:.4f}, p={p_value:.6f}')
        plt.xlabel('Hamming Distance (bits)')
        plt.ylabel('Frequency')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{BASE_DIR}/figures/hamming_histogram.png", dpi=300)
        plt.close()

    # --- Advanced Statistical Analysis ---

    if statistical_rigor and len(bit_vectors) > 1000:
        print("Performing advanced statistical analysis...")

        # Convert to bit array format for testing
        bit_array = np.stack(bit_vectors)

        # Run NIST-based tests
        test_results = {}
        nist_suite = NBISTTestSuite()

        for bit_pos in tqdm(range(160), desc="Testing bit positions"):
            # Extract single bit position across all samples
            bits = bit_array[:, bit_pos].astype(int).tolist()
            test_results[bit_pos] = nist_suite.run_all_tests(bits)

        # Save test results
        with open(f"{BASE_DIR}/reports/nist_test_results.json", 'w') as f:
            json.dump(test_results, f, indent=2)

        # Summarize results
        summary = {
            "failed_tests": 0,
            "total_tests": 0,
            "significant_bits": []
        }

        for bit_pos, results in test_results.items():
            for test_name, result in results.items():
                summary["total_tests"] += 1
                if not result["passed"]:
                    summary["failed_tests"] += 1
                    summary["significant_bits"].append({
                        "bit": bit_pos,
                        "test": test_name,
                        "p_value": result["p_value"]
                    })

        with open(f"{BASE_DIR}/reports/nist_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)

    return {
        'hash_samples': hash_samples,
        'bit_vectors': bit_vectors,
        'hamming_distribution': hamming_distribution
    }

# === Block 13: Differential Cryptanalysis Mode ===

def hamming_sensitivity_analysis(sample_size=1000, bit_positions=[0, 64, 128]):
    """Analyze how single-bit flips in private keys affect hash160 outputs"""
    print(f"Running differential analysis with {sample_size:,} samples per bit position...")

    sensitivity_results = {bit: [] for bit in bit_positions}

    for bit_pos in bit_positions:
        hamming_diffs = []

        for _ in tqdm(range(sample_size), desc=f"Bit {bit_pos}"):
            # Generate a random key
            k = random.randint(1, MAX_K)
            h160 = privkey_to_hash160(k)
            if not h160:
                continue

            # Flip one bit in the private key
            k_flipped = k ^ (1 << bit_pos)
            h160_flipped = privkey_to_hash160(k_flipped)
            if not h160_flipped:
                continue

            # Calculate Hamming distance
            ham, flips = hamming_distance_and_flips(h160, h160_flipped)
            hamming_diffs.append(ham)

            # Update bit bias for significant changes
            if ham < 80:  # Lower than expected for random (160 * 0.5)
                update_bit_bias(flips, reward=1.0/ham if ham > 0 else 1.0)

        sensitivity_results[bit_pos] = hamming_diffs

    # Visualize results
    plt.figure(figsize=(12, 8))
    for bit_pos, hamming_diffs in sensitivity_results.items():
        if hamming_diffs:
            plt.hist(hamming_diffs, bins=30, alpha=0.5, label=f'Bit {bit_pos}',
                    density=True)

    # Add expected binomial distribution
    x = np.arange(40, 120)
    n = 160
    p = 0.5
    expected = stats.binom.pmf(x, n, p)
    plt.plot(x, expected, 'k--', linewidth=2, label='Expected (Binomial)')

    plt.title('Hamming Distance Distribution for Single-Bit Flips')
    plt.xlabel('Hamming Distance')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/figures/hamming_sensitivity.png", dpi=300)
    plt.close()

    # Save statistical analysis
    analysis = {}
    for bit_pos, hamming_diffs in sensitivity_results.items():
        if hamming_diffs:
            analysis[bit_pos] = {
                "mean": np.mean(hamming_diffs),
                "std": np.std(hamming_diffs),
                "min": min(hamming_diffs),
                "max": max(hamming_diffs),
                "ks_test": stats.kstest(hamming_diffs, stats.binom.cdf, args=(160, 0.5))
            }

    with open(f"{BASE_DIR}/reports/sensitivity_analysis.json", 'w') as f:
        json.dump(analysis, f, indent=2)

    print("Differential analysis complete.")
    return sensitivity_results

# === Block 14: Command Line Interface and Research Mode Selection ===

def print_banner():
    """Display program banner and info"""
    banner = """
 ██▀███   ██▓ ██▓███  ▓█████  ███▄ ▄███▓ ██▓███   ██░ ██  ▄▄▄       ███▄    █ ▄▄▄█████▓ ▒█████   ███▄ ▄███▓
▓██ ▒ ██▒▓██▒▓██░  ██▒▓█   ▀ ▓██▒▀█▀ ██▒▓██░  ██▒▓██░ ██▒▒████▄     ██ ▀█   █ ▓  ██▒ ▓▒▒██▒  ██▒▓██▒▀█▀ ██▒
▓██ ░▄█ ▒▒██▒▓██░ ██▓▒▒███   ▓██    ▓██░▓██░ ██▓▒▒██▀▀██░▒██  ▀█▄  ▓██  ▀█ ██▒▒ ▓██░ ▒░▒██░  ██▒▓██    ▓██░
▒██▀▀█▄  ░██░▒██▄█▓▒ ▒▒▓█  ▄ ▒██    ▒██ ▒██▄█▓▒ ▒░▓█ ░██ ░██▄▄▄▄██ ▓██▒  ▐▌██▒░ ▓██▓ ░ ▒██   ██░▒██    ▒██
░██▓ ▒██▒░██░▒██▒ ░  ░░▒████▒▒██▒   ░██▒▒██▒ ░  ░░▓█▒░██▓ ▓█   ▓██▒▒██░   ▓██░  ▒██▒ ░ ░ ████▓▒░▒██▒   ░██▒
░ ▒▓ ░▒▓░░▓  ▒▓▒░ ░  ░░░ ▒░ ░░ ▒░   ░  ░▒▓▒░ ░  ░ ▒ ░░▒░▒ ▒▒   ▓▒█░░ ▒░   ▒ ▒   ▒ ░░   ░ ▒░▒░▒░ ░ ▒░   ░  ░
  ░▒ ░ ▒░ ▒ ░░▒ ░      ░ ░  ░░  ░      ░░▒ ░      ▒ ░▒░ ░  ▒   ▒▒ ░░ ░░   ░ ▒░    ░      ░ ▒ ▒░ ░  ░      ░
  ░░   ░  ▒ ░░░          ░   ░      ░   ░░        ░  ░░ ░  ░   ▒      ░   ░ ░   ░      ░ ░ ░ ▒  ░      ░
   ░      ░                ░  ░       ░             ░  ░  ░      ░  ░         ░              ░ ░         ░

    RIPEMD-160 ENTROPY ANALYSIS AND NON-UNIFORMITY RESEARCH PLATFORM (T5) v1.0

    • Comprehensive RIPEMD-160 Entropy Analysis Framework
    • Advanced Statistical Test Suite with Confidence Intervals
    • Avalanche Effect Quality Measurement System
    • Neural Pattern Recognition for Anomaly Detection
    • Differential Cryptanalysis Toolkit
    • LaTeX-Ready Scientific Report Generation

    Research Mode Options:
    1. Balanced - General research with equal focus on all aspects
    2. Entropy - Focus on statistical entropy analysis and anomaly detection
    3. Cluster - Focus on bit correlation and clustering patterns
    4. Targeted - Focus on specific bit patterns in target addresses
    5. Differential - Focus on avalanche effect and sensitivity analysis
    """
    print(banner)

def interactive_mode():
    """Present interactive menu and run selected research mode"""
    print_banner()

    print("\nRIPEMPHANTOM RESEARCH CONTROL PANEL")
    print("-" * 50)

    print("\nSelect a research mode:")
    print("1. Run full research with balanced approach (default)")
    print("2. Focus on entropy anomaly detection")
    print("3. Focus on bit correlation and clustering")
    print("4. Focus on target address pattern analysis")
    print("5. Run differential analysis of avalanche effect")
    print("6. Run standalone statistical test suite")
    print("7. Run pattern correlation analysis")
    print("8. Generate comprehensive report from existing data")
    print("9. Exit")

    while True:
        try:
            choice = int(input("\nEnter your selection (1-9): "))
            if 1 <= choice <= 9:
                break
            else:
                print("Invalid choice. Please enter a number between 1 and 9.")
        except ValueError:
            print("Please enter a valid number.")

    if choice == 9:
        print("Exiting RIPEMPHANTOM-T5.")
        return

    if choice == 1:
        research_mode = "balanced"
        print("\nSelected: Balanced Research Mode")
    elif choice == 2:
        research_mode = "entropy"
        print("\nSelected: Entropy Anomaly Detection Mode")
    elif choice == 3:
        research_mode = "cluster"
        print("\nSelected: Bit Correlation and Clustering Mode")
    elif choice == 4:
        research_mode = "targeted"
        print("\nSelected: Target Pattern Analysis Mode")
    elif choice == 5:
        # Run differential analysis directly
        print("\nRunning Differential Analysis Mode")
        sample_size = int(input("Enter sample size per bit position (recommended: 1000): "))
        hamming_sensitivity_analysis(sample_size=sample_size)
        return interactive_mode()  # Return to menu after completion
    elif choice == 6:
        # Run statistical test suite directly
        print("\nRunning Statistical Test Suite")
        sample_size = int(input("Enter sample size (recommended: 10000): "))
        run_entropy_scan(sample_size=sample_size, statistical_rigor=True)
        return interactive_mode()  # Return to menu after completion
    elif choice == 7:
        # Run pattern correlation analysis directly
        print("\nRunning Pattern Correlation Analysis")
        sample_size = int(input("Enter sample size per pattern (recommended: 2000): "))
        pattern_correlation_analysis(sample_size=sample_size)
        return interactive_mode()  # Return to menu after completion
    elif choice == 8:
        # Generate comprehensive report
        print("\nGenerating Comprehensive Report from Existing Data")
        create_comprehensive_report()
        print("Report generated successfully.")
        return interactive_mode()  # Return to menu after completion

    # For modes that require address files
    print("\nAddress file upload required for modes 1-4.")
    print("Upload your addresses.txt (one per line, base58 P2PKH)")

    # Wait for file upload
    files.upload()

    if not any(os.path.exists(p) for p in ["addresses.txt", "addresses (1).txt", "addresses (2).txt",
                                           "addresses (3).txt", "addresses (4).txt", "addresses (5).txt"]):
        print("ERROR: addresses.txt not uploaded. Aborting.")
        return interactive_mode()

    # Load addresses
    targets = load_addresses(ADDR_FILE)
    if not targets:
        print("No valid addresses found.")
        return interactive_mode()

    print(f"Loaded {len(targets)} addresses.")

    # Get steps
    max_steps = int(input("\nEnter maximum steps to run (recommended: 100,000): "))

    # Start research
    print(f"\nStarting {research_mode} research with {max_steps} steps...")
    result = run_phantom_t5(targets, max_steps=max_steps, research_mode=research_mode)

    print("\nResearch complete!")
    print(f"Best Hamming distance found: {result['best_hamming']}")
    print(f"Hash samples collected: {result['hash_samples']}")
    print(f"Anomalies found: {result['anomalies_found']}")

    return interactive_mode()  # Return to menu after completion

# === Block 15: Entry Point – Advanced Research Platform Launcher ===

if __name__ == "__main__":
    print("Upload your addresses.txt (one per line, base58 P2PKH) or press Enter for interactive mode:")

    try:
        uploaded = files.upload()

        if not uploaded and not any(os.path.exists(p) for p in ["addresses.txt", "addresses (1).txt",
                                                                "addresses (2).txt", "addresses (3).txt",
                                                                "addresses (4).txt", "addresses (5).txt"]):
            print("No file uploaded. Launching interactive mode...")
            interactive_mode()
        else:
            # Direct execution with uploaded file
            targets = load_addresses(ADDR_FILE)
            if not targets:
                print("No valid addresses found. Launching interactive mode...")
                interactive_mode()
            else:
                print(f"Loaded {len(targets)} addresses. Beginning scan...")
                run_phantom_t5(targets, max_steps=250_000)
                print("Research complete.")
    except Exception as e:
        print(f"Error during execution: {e}")
        print("Launching interactive mode...")
        interactive_mode()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.7 MB/s eta 0:00:00
Mounted at /content/drive
Upload your addr

No file uploaded. Launching interactive mode...

 ██▀███   ██▓ ██▓███  ▓█████  ███▄ ▄███▓ ██▓███   ██░ ██  ▄▄▄       ███▄    █ ▄▄▄█████▓ ▒█████   ███▄ ▄███▓
▓██ ▒ ██▒▓██▒▓██░  ██▒▓█   ▀ ▓██▒▀█▀ ██▒▓██░  ██▒▓██░ ██▒▒████▄     ██ ▀█   █ ▓  ██▒ ▓▒▒██▒  ██▒▓██▒▀█▀ ██▒
▓██ ░▄█ ▒▒██▒▓██░ ██▓▒▒███   ▓██    ▓██░▓██░ ██▓▒▒██▀▀██░▒██  ▀█▄  ▓██  ▀█ ██▒▒ ▓██░ ▒░▒██░  ██▒▓██    ▓██░
▒██▀▀█▄  ░██░▒██▄█▓▒ ▒▒▓█  ▄ ▒██    ▒██ ▒██▄█▓▒ ▒░▓█ ░██ ░██▄▄▄▄██ ▓██▒  ▐▌██▒░ ▓██▓ ░ ▒██   ██░▒██    ▒██ 
░██▓ ▒██▒░██░▒██▒ ░  ░░▒████▒▒██▒   ░██▒▒██▒ ░  ░░▓█▒░██▓ ▓█   ▓██▒▒██░   ▓██░  ▒██▒ ░ ░ ████▓▒░▒██▒   ░██▒
░ ▒▓ ░▒▓░░▓  ▒▓▒░ ░  ░░░ ▒░ ░░ ▒░   ░  ░▒▓▒░ ░  ░ ▒ ░░▒░▒ ▒▒   ▓▒█░░ ▒░   ▒ ▒   ▒ ░░   ░ ▒░▒░▒░ ░ ▒░   ░  ░
  ░▒ ░ ▒░ ▒ ░░▒ ░      ░ ░  ░░  ░      ░░▒ ░      ▒ ░▒░ ░  ▒   ▒▒ ░░ ░░   ░ ▒░    ░      ░ ▒ ▒░ ░  ░      ░
  ░░   ░  ▒ ░░░          ░   ░      ░   ░░        ░  ░░ ░  ░   ▒      ░   ░ ░   ░      ░ ░ ░ ▒  ░      ░   
   ░      ░                ░  ░       ░             ░  ░  ░      ░  ░         ░        

Saving addresses.txt to addresses.txt
Loaded 999 addresses.

Enter maximum steps to run (recommended: 100,000): 10000

Starting balanced research with 10000 steps...
Loaded consolidated state file from step 28000
Starting RIPEMPHANTOM-T5 with 999 targets and 10000 steps...


Research Progress: 0it [00:00, ?it/s]



Research complete!
Best Hamming distance found: 160
Hash samples collected: 0
Anomalies found: 0

 ██▀███   ██▓ ██▓███  ▓█████  ███▄ ▄███▓ ██▓███   ██░ ██  ▄▄▄       ███▄    █ ▄▄▄█████▓ ▒█████   ███▄ ▄███▓
▓██ ▒ ██▒▓██▒▓██░  ██▒▓█   ▀ ▓██▒▀█▀ ██▒▓██░  ██▒▓██░ ██▒▒████▄     ██ ▀█   █ ▓  ██▒ ▓▒▒██▒  ██▒▓██▒▀█▀ ██▒
▓██ ░▄█ ▒▒██▒▓██░ ██▓▒▒███   ▓██    ▓██░▓██░ ██▓▒▒██▀▀██░▒██  ▀█▄  ▓██  ▀█ ██▒▒ ▓██░ ▒░▒██░  ██▒▓██    ▓██░
▒██▀▀█▄  ░██░▒██▄█▓▒ ▒▒▓█  ▄ ▒██    ▒██ ▒██▄█▓▒ ▒░▓█ ░██ ░██▄▄▄▄██ ▓██▒  ▐▌██▒░ ▓██▓ ░ ▒██   ██░▒██    ▒██ 
░██▓ ▒██▒░██░▒██▒ ░  ░░▒████▒▒██▒   ░██▒▒██▒ ░  ░░▓█▒░██▓ ▓█   ▓██▒▒██░   ▓██░  ▒██▒ ░ ░ ████▓▒░▒██▒   ░██▒
░ ▒▓ ░▒▓░░▓  ▒▓▒░ ░  ░░░ ▒░ ░░ ▒░   ░  ░▒▓▒░ ░  ░ ▒ ░░▒░▒ ▒▒   ▓▒█░░ ▒░   ▒ ▒   ▒ ░░   ░ ▒░▒░▒░ ░ ▒░   ░  ░
  ░▒ ░ ▒░ ▒ ░░▒ ░      ░ ░  ░░  ░      ░░▒ ░      ▒ ░▒░ ░  ▒   ▒▒ ░░ ░░   ░ ▒░    ░      ░ ▒ ▒░ ░  ░      ░
  ░░   ░  ▒ ░░░          ░   ░      ░   ░░        ░  ░░ ░  ░   ▒      ░   ░ ░   ░      ░ ░ ░ ▒  ░      ░   
   ░      ░                ░  ░      

...

Research Progress:  83%|████████▎ | 33100/40000 [1:49:24<23:50,  4.82it/s]

Step 43100: Processed 2118400 hash samples, Best Hamming: 49


Research Progress:  83%|████████▎ | 33120/40000 [1:49:28<22:44,  5.04it/s]


Interrupted at step 43120. Saving checkpoint...
Checkpoint saved. Exiting gracefully.
Debug: Analyzing 1CpyTMbZcqRGLtbJnaijKRUu9hWrgsGK2D, x type: <class 'numpy.ndarray'>, y type: <class 'numpy.ndarray'>, len(x): 1000, len(y): 1000
Error in linregress for 1CpyTMbZcqRGLtbJnaijKRUu9hWrgsGK2D: 'dict' object has no attribute 'linregress'
Debug: Analyzing 1AEnZzB3quBNgJdYLZyvyHkoFyY1AFg8ZW, x type: <class 'numpy.ndarray'>, y type: <class 'numpy.ndarray'>, len(x): 1000, len(y): 1000
Error in linregress for 1AEnZzB3quBNgJdYLZyvyHkoFyY1AFg8ZW: 'dict' object has no attribute 'linregress'
Debug: Analyzing 15nSKG4BZ3th9zDsQ2nvCfS244eEAnys5a, x type: <class 'numpy.ndarray'>, y type: <class 'numpy.ndarray'>, len(x): 1000, len(y): 1000
Error in linregress for 15nSKG4BZ3th9zDsQ2nvCfS244eEAnys5a: 'dict' object has no attribute 'linregress'
Debug: Analyzing 1JFMfW5urn48S1o7fvBBSCu3WHX6SVinvm, x type: <class 'numpy.ndarray'>, y type: <class 'numpy.ndarray'>, len(x): 1000, len(y): 1000
Error in linregres

NameError: name 'create_comprehensive_report' is not defined